### Ocean Wind 1

The rest of this tutorial uses pre compiled ORBIT configs that are stored as .yaml files in the '~/configs/ folder. There are load and save methods available in ORBIT for working with .yaml files. These example projects each exhibit different functionalities within ORBIT. Using these examples and combinations of them, most project configurations can be modeled. 

In [1]:
import os
import pandas as pd
from ORBIT import ProjectManager, load_config

weather = pd.read_csv("data/example_weather.csv", parse_dates=["datetime"])\
            .set_index("datetime")

### Load the project configuration

In [2]:
fixed_config = load_config("configs/Ocean_Wind1.yaml")  # Configs can be loaded with absolute or relative paths

print(type(fixed_config))                                         # They are loaded in as dictionaries.
print(fixed_config)

print(f"Num turbines: {fixed_config['plant']['num_turbines']}")   # Once a configuration is loaded, different parameters can  
print(f"Turbine: {fixed_config['turbine']}")                      # be accessed using dict access.
print(f"\nSite: {fixed_config['site']}")


<class 'dict'>
{'site': {'depth': 26.381765177291, 'distance': 214, 'distance_to_landfall': 116, 'mean_windspeed': 9.16}, 'plant': {'layout': 'grid', 'num_turbines': 92, 'row_spacing': 8.613953488, 'substation_distance': 1, 'turbine_spacing': 2}, 'landfall': {'interconnection_distance': 11, 'trench_length': 2}, 'turbine': '12MW_generic', 'oss_design': {'num_substations': 3}, 'array_cable_install_vessel': 'example_cable_lay_vessel', 'export_cable_install_vessel': 'example_cable_lay_vessel', 'export_cable_bury_vessel': 'example_cable_lay_vessel', 'oss_install_vessel': 'example_heavy_lift_vessel', 'spi_vessel': 'example_scour_protection_vessel', 'wtiv': 'example_wtiv', 'OffshoreSubstationInstallation': {'feeder': 'example_heavy_feeder', 'num_feeders': 1}, 'array_system_design': {'cables': ['XLPE_630mm_66kV', 'XLPE_185mm_66kV']}, 'export_system_design': {'cables': 'XLPE_1000m_220kV', 'percent_added_length': 0.0}, 'scour_protection_design': {'cost_per_tonne': 40, 'scour_protection_depth': 1

### Phases

This fixed project represents a generic Offshore Wind farm with 50 6MW turbines. It includes 5 design modules and 6 installation modules as seen below. This is a common set of modules to run for a fixed bottom project. This config will model the procurement and installation of monopiles, scour protection, array system, export system, offshore substation and the turbines.

In [3]:
print(f"Design phases: {fixed_config['design_phases']}")
print(f"\nInstall phases: {list(fixed_config['install_phases'].keys())}")

Design phases: ['MonopileDesign', 'ScourProtectionDesign', 'ArraySystemDesign', 'ExportSystemDesign', 'OffshoreSubstationDesign']

Install phases: ['ArrayCableInstallation', 'ExportCableInstallation', 'MonopileInstallation', 'OffshoreSubstationInstallation', 'ScourProtectionInstallation', 'TurbineInstallation']


### Run

This project is always being modeled with the example weather project supplied that is representative of US East Coast wind farm locations.

In [4]:
project = ProjectManager(fixed_config, weather=weather)
print(f"\nProject Capacity: {project.config['plant']['capacity']}")
project.run()

ORBIT library intialized at 'c:\users\dmulash\documents\orbit\library'

Project Capacity: 1104


### Top Level Outputs

ProjectManager offers several high level result categories:
- Installation CapEx
- System CapEx (procurement of BOS subcomponents)
- Turbine CapEx
- Soft CapEx (project management costs)
- Total CapEx
- Total installation time
- etc.

In [5]:
print(f"Installation CapEx:  {project.installation_capex/1e6:.0f} M")
print(f"System CapEx:        {project.system_capex/1e6:.0f} M")
print(f"Turbine CapEx:       {project.turbine_capex/1e6:.0f} M")
print(f"Soft CapEx:          {project.soft_capex/1e6:.0f} M")
print(f"Total CapEx:        {project.total_capex/1e6:.0f} M")

print(f"\nInstallation Time: {project.installation_time:.0f} h")

print(f"\nCapEx per kW: {project.total_capex / (project.config['plant']['capacity'] * 1000):.0f} $/kW")

Installation CapEx:  531 M
System CapEx:        1180 M
Turbine CapEx:       1656 M
Soft CapEx:          600 M
Total CapEx:        4118 M

Installation Time: 36643 h

CapEx per kW: 3730 $/kW


### CapEx Breakdown

In [6]:
# The breakdown of project costs by module is available  at 'capex_breakdown'
project.capex_breakdown

{'Array System': 44368514.97425006,
 'Export System': 431889698.00168,
 'Offshore Substation': 171240450.0,
 'Scour Protection': 18241760,
 'Substructure': 514120861.30114037,
 'Array System Installation': 35592291.17363794,
 'Export System Installation': 221959939.79631707,
 'Offshore Substation Installation': 10798415.659240933,
 'Scour Protection Installation': 89029474.88584475,
 'Substructure Installation': 64786859.32048451,
 'Turbine Installation': 108995724.01666498,
 'Turbine': 1656000000,
 'Soft': 599648640.0,
 'Project': 151250000.0}

### Installation Actions

In [7]:
df = pd.DataFrame(project.actions)    # The project simulation logs are also available for all modules
df

,cost_multiplier,agent,action,duration,cost,level,time,phase,phase_name,max_waveheight,max_windspeed,transit_speed,location,site_depth,hub_height
0,0.5,Array Cable Installation Vessel,Mobilize,72.000000,180000.000000,ACTION,0.000000,ArrayCableInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.5,Heavy Lift Vessel,Mobilize,72.000000,750000.000000,ACTION,0.000000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.5,Feeder 0,Mobilize,72.000000,180000.000000,ACTION,0.000000,OffshoreSubstationInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.5,SPI Vessel,Mobilize,72.000000,180000.000000,ACTION,0.000000,ScourProtectionInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,SPI Vessel,Load SP Material,4.000000,20000.000000,ACTION,4.000000,ScourProtectionInstallation,ScourProtectionInstallation,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6133,NaN,WTIV,Attach Blade,3.500000,26250.000000,ACTION,16875.415171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
6134,NaN,WTIV,Release Blade,1.000000,7500.000000,ACTION,16876.415171,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6135,NaN,WTIV,Lift Blade,1.320000,9900.000000,ACTION,16877.735171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
6136,NaN,WTIV,Attach Blade,3.500000,26250.000000,ACTION,16881.235171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0


In [8]:
# These logs can be sorted by phase by using DataFrame operations

turbine_install = df.loc[df['phase']=="TurbineInstallation"]
turbine_install

,cost_multiplier,agent,action,duration,cost,level,time,phase,phase_name,max_waveheight,max_windspeed,transit_speed,location,site_depth,hub_height
1768,1.0,WTIV,Mobilize,168.000000,1.260000e+06,ACTION,6360.229418,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1769,NaN,WTIV,Fasten Tower Section,4.000000,3.000000e+04,ACTION,6364.229418,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
1772,NaN,WTIV,Fasten Tower Section,4.000000,3.000000e+04,ACTION,6368.229418,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
1773,NaN,WTIV,Fasten Nacelle,4.000000,3.000000e+04,ACTION,6372.229418,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
1774,NaN,WTIV,Fasten Blade,1.500000,1.125000e+04,ACTION,6373.729418,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6133,NaN,WTIV,Attach Blade,3.500000,2.625000e+04,ACTION,16875.415171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
6134,NaN,WTIV,Release Blade,1.000000,7.500000e+03,ACTION,16876.415171,TurbineInstallation,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6135,NaN,WTIV,Lift Blade,1.320000,9.900000e+03,ACTION,16877.735171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0
6136,NaN,WTIV,Attach Blade,3.500000,2.625000e+04,ACTION,16881.235171,TurbineInstallation,TurbineInstallation,NaN,NaN,NaN,NaN,26.381765,132.0


In [9]:
# Operations can also be grouped to see a total amount of time spend on each operation

turbine_install.groupby(["action"]).sum()['duration']

action
Attach Blade              966.000000
Attach Nacelle            552.000000
Attach Tower Section     1104.000000
Delay                    2231.000000
Fasten Blade              414.000000
Fasten Nacelle            368.000000
Fasten Tower Section      736.000000
Jackdown                   31.514149
Jackup                     31.514149
Lift Blade                364.320000
Lift Nacelle              121.440000
Lift Tower Section        182.160000
Mobilize                  168.000000
Position Onsite           184.000000
Reequip                   184.000000
Release Blade             276.000000
Release Nacelle           276.000000
Release Tower Section     552.000000
Transit                  1947.400000
Name: duration, dtype: float64